In [ ]:

# Step 2: Convert transcripts to Bopomofo

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from g2pw import G2PWConverter

OUT_DIR = Path("../npy")
META_PATH = OUT_DIR / "meta.csv"

BATCH_SIZE = 256


In [ ]:

# Load meta.csv generated in Step 1

meta = pd.read_csv(META_PATH)

print("rows:", len(meta))
print("unique sentences:", meta["sentence"].fillna("").astype(str).nunique())

display(meta.head())


In [ ]:

# Convert sentence -> Bopomofo.
#
# Duplicate sentences are converted only once.
# Conversion runs in batches.
# No manual pronunciation correction is applied here.

need_conversion = (
    "bopomofo" not in meta.columns
    or meta["bopomofo"].isna().any()
)

if need_conversion:
    converter = G2PWConverter()

    sentences = meta["sentence"].fillna("").astype(str)
    unique_sentences = sentences.unique().tolist()

    print("Converting unique sentences:", len(unique_sentences))

    unique_results = []

    for start in tqdm(
        range(0, len(unique_sentences), BATCH_SIZE),
        desc="Bopomofo conversion",
    ):
        batch = unique_sentences[start:start + BATCH_SIZE]
        unique_results.extend(converter(batch))

    sentence_to_bopomofo = {
        sentence: "".join(
            token for token in result
            if token is not None
        )
        for sentence, result in zip(unique_sentences, unique_results)
    }

    meta["bopomofo"] = sentences.map(sentence_to_bopomofo)
else:
    print("bopomofo already exists; conversion skipped.")

# Keep bopomofo immediately after sentence.
cols = meta.columns.tolist()
cols.remove("bopomofo")
sentence_pos = cols.index("sentence")
cols.insert(sentence_pos + 1, "bopomofo")
meta = meta[cols]

meta.to_csv(META_PATH, index=False)

print("saved:", META_PATH)
display(meta[["sentence", "bopomofo"]].head(10))


In [ ]:
# Check the generated Bopomofo column.

print("empty bopomofo rows:", int(meta["bopomofo"].astype(str).str.len().eq(0).sum()))
